In [ ]:
%sql
/* ===================================================================================
   EBA04 - Preview newly added rows in the combined table
   Rows in ritm16070584_combined_04272026 that do NOT exist in the original 7 files.
   Expensecategory is excluded from the comparison (formatting differs).
=================================================================================== */

WITH Old_Coupa_Union AS (
    SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered
    UNION ALL SELECT * FROM hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered
)

SELECT c.DateExpenseIncurrred, c.PublicOfficial, c.Expensecategory, c.Total,
       c.BusinessPurpose, c.Account, c.Divisor
FROM hive_metastore.ra_adido_2025.ritm16070584_combined_04272026 c
WHERE NOT EXISTS (
    SELECT 1 FROM Old_Coupa_Union o
    WHERE TRIM(c.DateExpenseIncurrred) <=> TRIM(o.DateExpenseIncurrred)
      AND TRIM(c.PublicOfficial) <=> TRIM(o.PublicOfficial)
      AND TRIM(c.Total) <=> TRIM(o.Total)
      AND TRIM(c.BusinessPurpose) <=> TRIM(o.BusinessPurpose)
      AND TRIM(c.Account) <=> TRIM(o.Account)
)
ORDER BY c.DateExpenseIncurrred, c.Account

In [ ]:
# ===================================================================================
# EXPORT 1: Newly added rows -> single CSV
# EXPORT 2: Reduced combined table (new rows removed) -> split CSVs (max 80k rows)
# ===================================================================================
from pyspark.sql.functions import col, trim, row_number
from pyspark.sql.window import Window
from functools import reduce
from operator import and_
import math

CHUNK_SIZE = 80000

def safe_rm(path):
    try:
        dbutils.fs.rm(path, recurse=True)
    except Exception:
        pass

def export_single_csv(df, output_dir, final_path):
    safe_rm(output_dir)
    safe_rm(final_path)
    df.coalesce(1).write.mode("overwrite").option("header", "true").csv(output_dir)
    part_file = [f.path for f in dbutils.fs.ls(output_dir) if f.name.startswith("part-")][0]
    dbutils.fs.cp(part_file, final_path)
    safe_rm(output_dir)
    return final_path

def export_chunked_csv(df, total_rows, base_path, chunk_size=CHUNK_SIZE):
    """Export a large DataFrame as multiple CSVs, each with at most chunk_size rows."""
    num_chunks = math.ceil(total_rows / chunk_size)
    w = Window.orderBy("DateExpenseIncurrred", "Account")
    df_numbered = df.withColumn("_row_num", row_number().over(w))
    exported = []
    for i in range(num_chunks):
        start = i * chunk_size + 1
        end = (i + 1) * chunk_size
        df_chunk = df_numbered.where((col("_row_num") >= start) & (col("_row_num") <= end)).drop("_row_num")
        part_num = i + 1
        chunk_path = base_path.replace(".csv", f"_part{part_num}.csv")
        staging_dir = base_path.replace(".csv", f"_staging_{part_num}")
        export_single_csv(df_chunk, staging_dir, chunk_path)
        chunk_count = df_chunk.count()
        exported.append((chunk_path, chunk_count))
        print(f"  Part {part_num}: {chunk_count} rows -> {chunk_path}")
    return exported

old_union_tables = [
    "hive_metastore.ra_adido_2025.ritm16070584_jan_feb2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_mar_april2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_may2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_june2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_jul_aug2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_sep_oct2025_filtered",
    "hive_metastore.ra_adido_2025.ritm16070584_nov_dec2024_filtered",
]

# Columns for matching (excluding Expensecategory due to format drift)
match_cols = ["DateExpenseIncurrred", "PublicOfficial", "Total", "BusinessPurpose", "Account"]

combined_table = "hive_metastore.ra_adido_2025.ritm16070584_combined_04272026"
df_combined = spark.table(combined_table)

# Build old union with TRIM-normalized match columns
df_old = None
for t in old_union_tables:
    df_t = spark.table(t).select(*[trim(col(c)).alias(c) for c in match_cols])
    df_old = df_t if df_old is None else df_old.unionAll(df_t)

# Build join condition
join_cond = [trim(df_combined[c]).eqNullSafe(df_old[c]) for c in match_cols]

# --- EXPORT 1: Newly added rows (NOT in old files) -> single CSV ---
print("=== EXPORT 1: Newly added rows ===")
df_new_rows = df_combined.join(df_old, reduce(and_, join_cond), "left_anti")
df_new_rows = df_new_rows.orderBy("DateExpenseIncurrred", "Account")

new_count = df_new_rows.count()
print(f"New rows found: {new_count}")
export_single_csv(df_new_rows, "dbfs:/tmp/eba04_new_rows_staging", "dbfs:/tmp/eba04_new_rows.csv")
print("Exported: dbfs:/tmp/eba04_new_rows.csv")

# --- EXPORT 2: Reduced combined table -> split CSVs (max 80k rows each) ---
print("\n=== EXPORT 2: Reduced combined table (split) ===")
df_reduced = df_combined.join(df_old, reduce(and_, join_cond), "left_semi")
df_reduced = df_reduced.orderBy("DateExpenseIncurrred", "Account")

reduced_count = df_reduced.count()
print(f"Reduced table rows: {reduced_count}")
print(f"Splitting into chunks of {CHUNK_SIZE} rows...")
export_chunked_csv(df_reduced, reduced_count, "dbfs:/tmp/eba04_reduced_combined.csv")

print(f"\n=== Summary ===")
print(f"  Combined table total:  {df_combined.count()}")
print(f"  New rows (1 file):     {new_count}")
print(f"  Reduced rows ({math.ceil(reduced_count/CHUNK_SIZE)} files): {reduced_count}")
print(f"  New + Reduced:         {new_count + reduced_count}")


In [ ]:
# ===================================================================================
# DOWNLOAD: Copy exported CSVs to FileStore for browser download
# After running, download each file at:
#   https://<your-databricks-instance>/files/eba04_exports/<filename>
# ===================================================================================
import re

dest_dir = "dbfs:/FileStore/eba04_exports"

# Clean destination
try:
    dbutils.fs.rm(dest_dir, recurse=True)
except Exception:
    pass
dbutils.fs.mkdirs(dest_dir)

# Collect all exported files from /tmp/
exported_files = []
for f in dbutils.fs.ls("dbfs:/tmp/"):
    if f.name.startswith("eba04_") and f.name.endswith(".csv"):
        exported_files.append(f)

exported_files.sort(key=lambda f: f.name)

# Copy each file and print download info
host = spark.conf.get("spark.databricks.workspaceUrl", "<your-databricks-instance>")

print(f"Copying {len(exported_files)} file(s) to {dest_dir}...\n")
for f in exported_files:
    dest_path = f"{dest_dir}/{f.name}"
    dbutils.fs.cp(f.path, dest_path)
    size_mb = round(f.size / (1024 * 1024), 2)
    download_url = f"https://{host}/files/eba04_exports/{f.name}"
    print(f"  {f.name}  ({size_mb} MB)")
    print(f"    -> {download_url}\n")

print("Done. Click the URLs above to download, or use:")
print(f'  dbutils.fs.cp("{dest_dir}/<filename>", "file:/tmp/<filename>")')
